# Test STBFM on a TSMC2014 NYC subset

This notebook loads a bounded subset of the preprocessed NYC check-in dataset, prepares local meter coordinates, and runs the Python STBFM implementation. The miner returns all PI-strong spatio-temporal sequential patterns that pass the participation-index threshold; it does not filter for closed patterns.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "Notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "Source"))

from stbfm import STBFM, add_local_meter_coordinates, patterns_to_frame

In [ ]:
data_path = PROJECT_ROOT / "Data" / "Preprocessed" / "dataset_TSMC2014_NYC_preprocessed.csv"

raw_data = pd.read_csv(data_path)
common_event_types = raw_data["Event_type"].value_counts().head(20).index
events = raw_data[raw_data["Event_type"].isin(common_event_types)].head(1000).copy()
events = add_local_meter_coordinates(events)

events[["Event_ID", "Event_type", "timestamp", "longitude", "latitude", "x_meters", "y_meters"]].head()

In [ ]:
events.shape, events["Event_type"].nunique(), events["Event_type"].value_counts()

In [ ]:
miner = STBFM(
    events,
    spatial_radius=500.0,
    temporal_window=60.0,
    min_participation_index=0.01,
    max_length=None,
)

patterns = miner.mine()
patterns_frame = patterns_to_frame(patterns)
patterns_frame

In [ ]:
assert not patterns_frame.empty
assert patterns_frame["participation_index"].min() > 0.05
assert patterns_frame["length"].min() >= 2

patterns_frame.head(10)

In [ ]:
patterns_frame.groupby("length").agg(
    patterns=("sequence", "count"),
    max_pi=("participation_index", "max"),
    min_pi=("participation_index", "min"),
).reset_index()